# Figure S9 histology of Huang et al. (2025)

**Paper.** Huang et al. (2025), *iScience* 28, 112104,
[doi:10.1016/j.isci.2025.112104](https://doi.org/10.1016/j.isci.2025.112104). The claims file was
written from the version of record in Europe PMC (PMC11930745).

**Data.** The dandiset (DANDI:001349, version 0.250604.1510) holds no histology. The paper's code
repository ([EdnaHuang/SubnetworkAssembley2024](https://github.com/EdnaHuang/SubnetworkAssembley2024)
at commit 28e1b6d) holds the Figure S9 counts, `Statistical_Script_Data/Analysis_VGAT-Syt2/data/P15_VGAT_SYT2_20230318.xlsx`,
which the claims file names for the two Figure S9 claims (`data: elsewhere`). The sheets read are
those the repository's R scripts read: `Sorted-ForMultilevel` (neuropil puncta densities per image)
and `Somatic_input_ForMultilevel` (per soma).

**Test.** The model of `VGAT_Syt2_Function_20230319.R`: y ~ layer * sex with random intercepts for
animal and animal by image (dropped when singular), and a case bootstrap over animals with 10,000
replicates; contrasts are averaged over the other factor, as emmeans does, and also given within
each level. As in the imaging notebook, each replicate refits the fixed effects by generalized least
squares with the variance components of the original fit.

| Claim | Tested here |
|---|---|
| figs9-no-sex-difference-gabaergic-input | yes |
| figs9-layer-differences | yes |

In [ ]:
import numpy as np
import pandas as pd
import subnet

neuropil, soma = subnet.histology_tables()
print("repository", subnet.REPO, "commit", subnet.REPO_COMMIT[:7])
print(f"neuropil images: {len(neuropil)}; somata: {len(soma)}")
for name, d in (("neuropil", neuropil), ("soma", soma)):
    print(f"\n{name}: animals by sex", d.groupby("sex").subject.nunique().to_dict(), "; rows by layer and sex")
    print(pd.crosstab(d.layer, d.sex).to_string())
    print(d.drop(columns=["subject", "sex", "layer"]).describe().T[["count", "min", "50%", "max"]].round(2).to_string())
print("\nUnits: densities in puncta per 100 um3; soma volume in um3; puncta on soma as counts, and as the normalized column of the sheet, whose unit it does not state.")

In [ ]:
fits = {}
def test(df, y):
    if y not in fits:
        fits[y] = subnet.histology_test(df, y, B=10000, seed=subnet.seed_for(y))
    return fits[y]

def contrasts(df, y, factor, a, b):
    r = test(df, y)
    rows = [{"measure": y, "contrast": f"{a} - {b}", "within": "averaged", **subnet.averaged_contrast(r, factor, a, b)}]
    other = "sex" if factor == "layer" else "layer"
    for lv in sorted(df[other].unique()):
        i = np.where((r["grid"][factor] == a) & (r["grid"][other] == lv))[0][0]
        j = np.where((r["grid"][factor] == b) & (r["grid"][other] == lv))[0][0]
        bt = r["boot"][:, i] - r["boot"][:, j]
        bt = bt[np.isfinite(bt)]
        rows.append({"measure": y, "contrast": f"{a} - {b}", "within": lv, "estimate": r["est"][i] - r["est"][j],
                     "lo": np.percentile(bt, 2.5), "hi": np.percentile(bt, 97.5),
                     "p": 2 * (1 - max(np.mean(bt > 0), np.mean(bt < 0)))})
    print(f"{y}: random structure {r['structure']}, {r['n']} rows of {r['n_subjects']} animals, {r['dropped']} replicates rank-deficient")
    return pd.DataFrame(rows)

## figs9-no-sex-difference-gabaergic-input

> At P15 the densities of GABAergic synapses in the neuropil and around somata show no sex difference.

A claim of absence: every sex contrast (VGAT, Syt2 and VGAT/Syt2 densities in the neuropil, and the
perisomatic VGAT/Syt2 density) has p >= 0.05.

In [ ]:
tab = pd.concat([contrasts(neuropil, y, "sex", "M", "F") for y in ("vgat", "syt2", "vgat_syt2")] +
                [contrasts(soma, "vgat_syt2_on_soma_norm", "sex", "M", "F")])
print(tab.round(4).to_string(index=False))
print("sex contrasts with p < 0.05:", int((tab.p < 0.05).sum()), "of", len(tab))

**Result: reproduced.** None of the 12 sex contrasts has p < 0.05: male minus female, averaged over
layers, is 0.85 (p = 0.33) for VGAT, 0.53 (p = 0.34) for Syt2 and 0.63 (p = 0.33) for VGAT/Syt2
densities in the neuropil, and -0.57 (p = 0.16) for the perisomatic VGAT/Syt2 density; within each
layer p is 0.26 to 0.54. The repository's output printed p from 0.103 to 0.28.

## figs9-layer-differences

> L4 has a higher density of Syt2 (putative parvalbumin) synapses than L2/3 but not of total VGAT
> synapses, and L4 spiny stellate neurons have smaller somata and fewer perisomatic GABAergic inputs
> than L2/3 neurons.

Syt2, soma volume and perisomatic inputs (VGAT puncta on the soma) must differ in the stated
direction with p < 0.05; the VGAT layer contrast is a claim of absence (p >= 0.05).

In [ ]:
tab = pd.concat([contrasts(neuropil, "syt2", "layer", "L4", "L2/3"), contrasts(neuropil, "vgat", "layer", "L4", "L2/3"),
                 contrasts(soma, "soma_volume", "layer", "L4", "L2/3"), contrasts(soma, "vgat_on_soma", "layer", "L4", "L2/3"),
                 contrasts(soma, "vgat_syt2_on_soma_norm", "layer", "L4", "L2/3")])
print(tab.round(4).to_string(index=False))
avg = tab[tab.within == "averaged"].set_index("measure")
parts = {"Syt2 higher in L4": avg.loc["syt2", "estimate"] > 0 and avg.loc["syt2", "p"] < 0.05,
         "soma smaller in L4": avg.loc["soma_volume", "estimate"] < 0 and avg.loc["soma_volume", "p"] < 0.05,
         "fewer perisomatic VGAT puncta in L4": avg.loc["vgat_on_soma", "estimate"] < 0 and avg.loc["vgat_on_soma", "p"] < 0.05,
         "no VGAT layer difference": avg.loc["vgat", "p"] >= 0.05}
print(parts)

**Result: partially reproduced.** L4 has more Syt2 puncta than L2/3 (5.42 per 100 um3, p < 0.0001),
smaller somata (-494 um3, p < 0.0001) and fewer VGAT puncta on the soma (-51.4, p < 0.0001), in both
sexes. The VGAT density is also higher in L4 (0.96, 0.55 to 1.39, p < 0.0001), as the claims file
notes the repository's output found (0.95), so the VGAT part of the claim does not hold.

## Summary

| Claim | Outcome |
|---|---|
| figs9-no-sex-difference-gabaergic-input | reproduced |
| figs9-layer-differences | partially reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

These claims use the paper's histology counts from its code repository, not the dandiset, which
holds none. The test differs from the paper's in refitting the fixed effects by generalized least
squares in each bootstrap replicate.